# Aula 3 — PID pelo lugar das raízes, relé e compensação do atrito

**Projeto final · Laboratório de Controle Automático · Ifes Guarapari · 25/11 · 1h40**

**Objetivo da aula:** corrigir o que a iteração 1 mostrou — o atrito prende o eixo — com três
ferramentas: um **PID mais rápido** projetado pelo lugar das raízes, uma **verificação** pelo
experimento do relé (Ziegler-Nichols) e a **compensação da zona morta** no firmware. Tudo é
testado primeiro no gêmeo digital e, no fim, na bancada.

**Ao final desta aula a bancada terá:** os ganhos finais ($k_p$, $k_i$, $k_d$), `ZM` e `EM`
escolhidos e justificados · a comparação com o Ziegler-Nichols · o primeiro ensaio do controlador
final. **Começa o Trabalho 2.**

**Papéis na bancada (troquem a cada aula):** 🔧 **operador(a)** — mexe no hardware e grava o
firmware · 💻 **programador(a)** — roda e edita o notebook · ✍️ **relator(a)** — escreve as
respostas ❓ e confere os números. Com 2 pessoas, o relator acumula com o programador.

**Legenda:** ⏱️ tempo sugerido · 📘 teoria (leia antes de fazer) · 🔧 bancada · 💻 notebook ·
✏️ complete o código · ❓ responda · ✅ checkpoint — **chame o professor**.

| ⏱️ | Parte | O que se faz |
|---|---|---|
| 0–40 min | 1 | PID pelo lugar das raízes — iteração 2 |
| 40–55 min | 2 | Experimento do relé e Ziegler-Nichols |
| 55–85 min | 3 | Compensação da zona morta e robustez (gêmeo) |
| 85–95 min | 4 | Primeiro ensaio do controlador final |
| 95–100 min | ✅ | Checkpoint 3 |

In [ ]:
# ⚙️ CONFIGURAÇÃO — rode esta célula sempre que abrir o notebook
import numpy as np
import matplotlib.pyplot as plt
import control as ct
from scipy.optimize import minimize
from ferramentas_projeto import *

PORTA = "COM5"     # ✏️ porta do Arduino (Arduino IDE > Ferramentas > Porta). Linux: "/dev/ttyUSB0" ou "/dev/ttyACM0"
BANCADA = "1"      # ✏️ número da bancada

s = ct.tf("s")
Ts = 0.02          # período de amostragem do firmware [s]
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})
P = carregar_parametros()          # resultados das aulas anteriores (dados/parametros_bancada.json)
print("bancada", BANCADA, "| parâmetros já salvos:", list(P))

In [ ]:
# modelo e gêmeo salvos na Aula 2
GEMEO = P["GEMEO"]
Km, tau = P["Km"], P["tau"]
L_ef = GEMEO["L"] + Ts / 2
G = Km / (s * (tau * s + 1))
Pd = ct.tf(*ct.pade(L_ef, 3))           # atraso aproximado por Padé (para LGR e margens)
print(f"G_p(s) = {Km:.3f} / (s({tau:.3f}s + 1)) · e^(-{1e3*L_ef:.1f} ms · s)")

## Parte 1 — PID pelo lugar das raízes: iteração 2 · ⏱️ 40 min

📘 **Por que mais rápido.** Na iteração 1, $k_p$ ≈ 0,7 e a faixa em que o atrito prende o eixo
era $F_s/k_p$ ≈ 60°. Um projeto com polos **mais à esquerda** (mais rápido) exige mais ganho e
encolhe essa faixa. Metas da iteração 2: $\zeta$ = 0,7 ($M_p$ ≈ 5 % no protótipo — sobra folga
para o efeito dos zeros) e $t_s$ = 0,3 s.

**💻 Passo 1.1** — polo desejado e ângulo que falta (o mesmo cálculo da Aula 2).

In [ ]:
zeta, ts_alvo = 0.7, 0.3
sigma = 3 / ts_alvo
wd = sigma * np.tan(np.arccos(zeta))
sd = complex(-sigma, wd)
Gsd = complex(ct.evalfr(G, sd)) * np.exp(-L_ef * sd)      # G(s_d) com o atraso
ang_G = np.degrees(np.angle(Gsd))
ang_G = ang_G - 360 if ang_G > 0 else ang_G
avanco = -180 - ang_G
print(f"s_d = {sd.real:.2f} + j{sd.imag:.2f}")
print(f"∠G(s_d) = {ang_G:.1f}°  (o atraso contribui com {-np.degrees(wd*L_ef):.1f}°)")
print(f"ângulo que o controlador precisa FORNECER: {avanco:.1f}°")

📘 **Avanço de fase com um PD.** O controlador $C_{PD}(s) = k_d\,(s + a)$ tem um **zero** em
$s = -a$. Em $s_d$, o vetor zero → $s_d$ contribui com o ângulo
$\theta_z = \arctan\!\big(\omega_d/(a - \sigma)\big)$. Para fornecer exatamente o ângulo $\varphi$
que falta, basta pôr o zero em
$$a = \sigma + \frac{\omega_d}{\tan\varphi}.$$
O ganho $k_d$ sai da condição de módulo: $k_d = 1\,/\,\big|(s_d + a)\,G(s_d)\,e^{-L_{ef}s_d}\big|$.

📘 **PID = PD × PI.** Juntando o PD com um PI de zero $z$ perto da origem:
$$C(s) = k_d\,\frac{(s + a)(s + z)}{s} = k_d\,s + k_d(a + z) + \frac{k_d\,a\,z}{s}
\quad\Rightarrow\quad k_p = k_d(a + z),\quad k_i = k_d\,a\,z.$$
Aqui usamos $z = \sigma/20$ (ainda mais perto da origem que na Aula 2): o integrador só precisa
"limpar" o resíduo do atrito, e integrador forte + atrito estático faz o eixo oscilar devagar em
torno da referência (*stick-slip*).

📘 **PID de verdade (o que o firmware faz).** A derivada pura amplifica o ruído do encoder; o
firmware usa a derivada **filtrada** $k_d\,\dfrac{N s}{s + N}$ com $N$ = 30 rad/s, e a aplica só
à **medição** (não ao erro), para não dar um "chute" no PWM a cada degrau de referência.

**💻 Passo 1.2** — complete as fórmulas ✏️.

In [ ]:
phi = np.radians(avanco)
a_pd = ...                # ✏️ posição do zero do PD:  σ + ωd / tan(φ)
kd = 1 / abs((sd + a_pd) * Gsd)                 # condição de módulo
z = sigma / 20
kp = ...                  # ✏️ kd·(a + z)
ki = ...                  # ✏️ kd·a·z
print(f"zero do PD em −{a_pd:.2f};  zero do PI em −{z:.3f}")
print(f"kp = {kp:.3f} PWM/grau   ki = {ki:.3f} PWM/(grau·s)   kd = {kd:.4f} PWM·s/grau")

In [ ]:
desenha_lgr((s + a_pd) * G * Pd, sd, kmax=1.0,
            titulo=r"LGR de $k_d(s+a)\,G_p(s)\,e^{-L_{ef}s}$ — o ramo tem que passar por $s_d$")
plt.show()
it2 = analisa_pid(G, L_ef, kp, ki, kd, gemeo=GEMEO, nome="Iteração 2 (sem compensação)")

❓ **1.1** (a) O LGR passa por $s_d$? (b) Compare $k_p$ com o da iteração 1 e calcule a nova faixa
$F_s/k_p$. (c) A margem de fase está entre 45° e 60°? (d) No gêmeo, o erro de regime já atende à
especificação 1?

✍️ **Resposta da bancada:** *(escreva aqui)*

## Parte 2 — Experimento do relé e Ziegler-Nichols · ⏱️ 15 min

📘 **Relé.** Trocamos o controlador por uma chave: PWM = $+d$ se o erro for positivo e $-d$ se for
negativo (com uma histerese $h$ para não trepidar). A malha **oscila** num ciclo de amplitude $a$ e
período $P_u$ que **não cresce**. O relé se comporta, para esse ciclo, como um ganho equivalente
$4d/(\pi a)$ — e a malha está exatamente no limite da estabilidade. Daí saem o **ganho último** e o
**período último**:
$$K_u = \frac{4d}{\pi\sqrt{a^2 - h^2}},\qquad P_u = \text{período da oscilação}.$$
📘 **Ziegler-Nichols (PID):** $k_p = 0{,}6K_u$, $T_i = P_u/2$, $T_d = P_u/8$, isto é,
$k_i = k_p/T_i$ e $k_d = k_p\,T_d$. É uma sintonia sem modelo — útil para conferir o projeto.

**🔧💻 Passo 2.1** — ensaio do relé (6 s; a roda vai "tremer" em torno de 0°).

In [ ]:
d_rele, h_rele = 120, 1.0
rele = coletar("rele", PORTA, MODO=2, D=d_rele, H=h_rele, CPG=P["CPG"])
m = rele["t"] > 2.0                                   # descarta o início
yr, tr_ = rele["angle"][m], rele["t"][m]
a_rele = (yr.max() - yr.min()) / 2
yc = yr - yr.mean()
subidas = tr_[1:][(yc[:-1] < 0) & (yc[1:] >= 0)]
Pu = float(np.mean(np.diff(subidas)))
Ku = 4 * d_rele / (np.pi * np.sqrt(a_rele**2 - h_rele**2))
plt.figure(figsize=(8, 2.8)); plt.plot(rele["t"], rele["angle"]); plt.xlabel("tempo (s)")
plt.ylabel("ângulo (graus)"); plt.title("Relé: ciclo-limite"); plt.show()
print(f"a = {a_rele:.2f}°   Pu = {Pu:.3f} s   Ku = {Ku:.2f} PWM/grau")

In [ ]:
kp_zn = 0.6 * Ku
ki_zn = ...               # ✏️ kp / Ti, com Ti = Pu/2
kd_zn = ...               # ✏️ kp · Td, com Td = Pu/8
zn = analisa_pid(G, L_ef, kp_zn, ki_zn, kd_zn, nome="Ziegler-Nichols")
plt.figure(figsize=(8, 3))
plt.plot(it2["t_lin"], it2["y_lin"], label="LGR (iteração 2)")
plt.plot(zn["t_lin"], zn["y_lin"], label="Ziegler-Nichols")
plt.axhline(90 * 1.15, color="r", ls=":", lw=1, label="limite de Mp")
plt.xlim(0, 3); plt.ylim(0, 200); plt.xlabel("tempo (s)"); plt.ylabel("ângulo (graus)")
plt.legend(); plt.title("Degrau de 90° no modelo linear"); plt.show()

❓ **2.1** Compare os ganhos e as margens do Ziegler-Nichols com os do LGR. O ZN serviria para esta
planta? Qual dos três ganhos mais difere — e por quê, lembrando que a planta já tem um integrador?

✍️ **Resposta da bancada:** *(escreva aqui)*

## Parte 3 — Compensação da zona morta e robustez · ⏱️ 30 min

📘 **Compensar a zona morta.** Se sabemos que o motor "perde" ≈ $F_c$ de PWM para o atrito, o
firmware pode **somar** esse valor ao comando, no sentido do comando:
$$u = u_{PID} + \text{ZM}\cdot\text{sinal}(u_{PID})\qquad\text{quando } |e| > \text{EM}.$$
Assim o PID "enxerga" um motor quase sem zona morta. Dois cuidados:
- `ZM` um pouco **abaixo** da menor zona morta medida (≈ 95 %): compensar demais faz o motor dar
  trancos;
- perto de $e = 0$, o sinal de $u_{PID}$ troca a toda hora e o PWM pularia entre $+$ZM e $-$ZM. Por
  isso há uma **faixa morta** `EM` (≈ 1 resolução do encoder) em que só o PID atua.

**💻 Passo 3.1** — escolha `ZM` e `EM` e compare três configurações no gêmeo.

In [ ]:
ZM = ...                  # ✏️ ≈ 95 % da MENOR zona morta medida (P["Fc_pos"], P["Fc_neg"]); número inteiro
EM = ...                  # ✏️ ≈ 1 resolução do encoder (1/CPG), arredondada para cima em 0,1°
GANHOS = dict(KP=kp, KI=ki, KD=kd, N=30)
cfgs = {"(a) sem compensação": dict(ZM=0, EM=0),
        "(b) ZM, sem faixa EM": dict(ZM=ZM, EM=0),
        "(c) ZM + faixa EM": dict(ZM=ZM, EM=EM)}
sims = {}
for nome, c in cfgs.items():
    sims[nome] = simular(GEMEO, MODO=3, **GANHOS, **c)
    imprime_metricas(metricas_ensaio(sims[nome]), f"\nGÊMEO — {nome}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5.4), sharex=True)
for (nome, d), cor in zip(sims.items(), ["tab:red", "tab:orange", "tab:blue"]):
    ax[0].plot(d["t"], d["angle"], color=cor, label=nome)
    ax[1].plot(d["t"], d["pwm"], color=cor, lw=1)
ax[0].axhline(90, color="k", ls="--", lw=0.8); ax[0].axhspan(88, 92, color="0.9", zorder=0)
ax[0].set_ylim(75, 100); ax[0].set_ylabel("ângulo (graus) — zoom"); ax[0].legend(fontsize=8)
ax[1].set_ylim(-120, 120); ax[1].set_ylabel("PWM"); ax[1].set_xlabel("tempo (s)"); ax[1].set_xlim(1.8, 7)
ax[0].set_title("Degrau 0 → 90° no gêmeo (faixa cinza = ±2°)"); plt.show()

📘 **Robustez.** O atrito muda de um ensaio para outro (temperatura, posição da engrenagem). Uma
configuração que funciona só para o atrito exato que medimos é frágil. `robustez()` sorteia 20
gêmeos com o atrito ±15 % diferente e conta em quantos as especificações 1–3 são atendidas nos
três degraus. (Leva ≈ 1 min.)

**💻 Passo 3.2**

In [ ]:
for nome, c in [("(a) sem compensação", dict(ZM=0, EM=0)), ("(c) ZM + faixa EM", dict(ZM=ZM, EM=EM))]:
    print(f"{nome}: {robustez(GEMEO, n=20, **GANHOS, **c)}/20 plantas atendem às especificações 1–3")

❓ **3.1** (a) O que a compensação mudou no erro de regime? (b) Olhe o PWM da configuração (b)
depois que o eixo para: o que acontece, e por que a faixa `EM` resolve? (c) O que o teste de
robustez diz sobre a configuração (c)?

✍️ **Resposta da bancada:** *(escreva aqui)*

## Parte 4 — Primeiro ensaio do controlador final · ⏱️ 10 min

**🔧💻 Passo 4.1** — o perfil completo na bancada, com os ganhos e a compensação escolhidos.

In [ ]:
CFG_FINAL = dict(MODO=3, CPG=P["CPG"], KP=round(kp, 3), KI=round(ki, 3), KD=round(kd, 4), N=30,
                 ZM=ZM, EM=EM, PWM_MAX=255, AW=1)
mf = coletar("mf_teste", PORTA, **CFG_FINAL)
sim = simular(GEMEO, **CFG_FINAL)
imprime_metricas(metricas_ensaio(mf), "BANCADA — controlador final (1º ensaio)")
plota_ensaio(sim, mf, rotulos=["gêmeo", "bancada"], titulo="Controlador final")
plt.show()
P = salvar_parametros(CFG_FINAL=CFG_FINAL, ZN=dict(kp=kp_zn, ki=ki_zn, kd=kd_zn, Ku=Ku, Pu=Pu))

## ✅ Checkpoint 3 · ⏱️ 5 min

Chame o professor com o gráfico acima na tela. Se alguma especificação falhou na bancada,
**anote qual e com quanto** — a Aula 4 começa por aí.

In [ ]:
c = P["CFG_FINAL"]
print(f"Bancada {BANCADA} — controlador final")
print(f"  kp = {c['KP']}  ki = {c['KI']}  kd = {c['KD']}  N = {c['N']}  ZM = {c['ZM']}  EM = {c['EM']}")
print(f"  Ziegler-Nichols: kp = {P['ZN']['kp']:.2f}  ki = {P['ZN']['ki']:.1f}  kd = {P['ZN']['kd']:.3f}")